In [ ]:
!pip uninstall -y trl sentence-transformers
!pip install -q transformers peft bitsandbytes accelerate datasets


In [ ]:
!pip install torch

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import torch
import transformers
import peft

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
)
from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
)

print("Transformers:", transformers.__version__)
print("PEFT:", peft.__version__)
print("Torch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())


Transformers: 4.57.6
PEFT: 0.18.1
Torch: 2.9.0+cu126
CUDA: True


In [ ]:
raw_dataset = load_dataset("zou-lab/MedCaseReasoning")


In [ ]:
def build_text(example):
    return {
        "text": (
            "You are a medical expert.\n\n"
            "Task:\n"
            "1. Identify the most likely diagnosis.\n"
            "2. List ALL clinical reasons used to reach the diagnosis.\n"
            "3. Be exhaustive. Do not summarize.\n\n"
            "Case:\n"
            f"{example['case_prompt']}\n\n"
            "Final Diagnosis:\n"
            f"{example['final_diagnosis']}\n\n"
            "Clinical Reasons (exhaustive list):\n"
            f"{example['diagnostic_reasoning']}\n\n"
            "Differential Diagnoses:\n"
        )
    }

train_text = raw_dataset["train"].map(
    build_text,
    remove_columns=raw_dataset["train"].column_names,
)

In [ ]:
MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"

from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
)

tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"



In [ ]:
MAX_LENGTH = 1024

def tokenize(batch):
    enc = tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH,
    )
    enc["labels"] = enc["input_ids"].copy()
    return enc

tokenized_train = train_text.map(
    tokenize,
    batched=True,
    remove_columns=["text"],
)


In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

from transformers import AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

model.config.use_cache = False
model = prepare_model_for_kbit_training(model)



In [ ]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.enable_input_require_grads()
model.print_trainable_parameters()


trainable params: 4,358,144 || all params: 1,548,072,448 || trainable%: 0.2815


In [ ]:
from transformers import TrainingArguments, Trainer
from transformers.trainer_utils import get_last_checkpoint
import os

# Output directory on Google Drive
OUTPUT_DIR = "/content/drive/MyDrive/biomedlm-qlora-rr"

# Training arguments
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,

    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,

    learning_rate=2e-4,
    num_train_epochs=1,

    fp16=True,

    logging_steps=50,

    save_steps=500,
    save_strategy="steps",
    save_total_limit=2,

    report_to="none",

    gradient_checkpointing=False,  # MUST be False for 4-bit QLoRA
)

# Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
)

# 🔑 Auto-resume logic (SAFE)
last_checkpoint = get_last_checkpoint(OUTPUT_DIR)

if last_checkpoint is not None:
    print(f"Resuming training from checkpoint: {last_checkpoint}")
    trainer.train(resume_from_checkpoint=last_checkpoint)
else:
    print("No checkpoint found. Starting training from scratch.")
    trainer.train()


No checkpoint found. Starting training from scratch.


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1044: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
50,2.608400
100,0.997900
150,0.964400
200,0.985500
250,0.951600
300,0.945700
350,0.946400
400,0.945800
450,0.949800
500,0.942500


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1044: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1044: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/pyt

In [ ]:
final_adapter_path = get_last_checkpoint(OUTPUT_DIR)

model = model.merge_and_unload()
model.save_pretrained("/content/drive/MyDrive/qwen-med-final")
tokenizer.save_pretrained("/content/drive/MyDrive/qwen-med-final")


/usr/local/lib/python3.12/dist-packages/peft/tuners/lora/bnb.py:397: UserWarning: Merge lora module to 4-bit linear may get different generations due to rounding errors.
  warnings.warn(


('/content/drive/MyDrive/qwen-med-final/tokenizer_config.json',
 '/content/drive/MyDrive/qwen-med-final/special_tokens_map.json',
 '/content/drive/MyDrive/qwen-med-final/chat_template.jinja',
 '/content/drive/MyDrive/qwen-med-final/vocab.json',
 '/content/drive/MyDrive/qwen-med-final/merges.txt',
 '/content/drive/MyDrive/qwen-med-final/added_tokens.json',
 '/content/drive/MyDrive/qwen-med-final/tokenizer.json')

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
from transformers.trainer_utils import get_last_checkpoint

BASE_MODEL_ID = "Qwen/Qwen2.5-1.5B"
OUTPUT_DIR = "/content/drive/MyDrive/biomedlm-qlora-rr"
FINAL_DIR = "/content/drive/MyDrive/qwen2.5-med-final"

tokenizer = AutoTokenizer.from_pretrained(
    BASE_MODEL_ID,
    trust_remote_code=True,
)
tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
)


In [ ]:
last_ckpt = get_last_checkpoint(OUTPUT_DIR)
print("Using checkpoint:", last_ckpt)

model = PeftModel.from_pretrained(
    base_model,
    last_ckpt,
)


Using checkpoint: /content/drive/MyDrive/biomedlm-qlora-rr/checkpoint-3273


In [ ]:
merged_model = model.merge_and_unload()

merged_model.save_pretrained(
    FINAL_DIR,
    safe_serialization=False  # forces pytorch_model.bin
)

tokenizer.save_pretrained(FINAL_DIR)

print("✅ Merged model saved to:", FINAL_DIR)


✅ Merged model saved to: /content/drive/MyDrive/qwen2.5-med-final


In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM


In [ ]:
MODEL_DIR = "/content/drive/MyDrive/qwen2.5-med-final"
DEVICE = "cuda"


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_DIR,
    trust_remote_code=True,
    fix_mistral_regex=True,   # suppress tokenizer warning
)

tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"


In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    MODEL_DIR,
    torch_dtype=torch.float16,
    device_map={"": 0},        # force all weights onto GPU
    low_cpu_mem_usage=True,    # prevents staging copies
    trust_remote_code=True,
)

model.eval()
torch.cuda.synchronize()

print("✅ Model loaded fully on GPU")


`torch_dtype` is deprecated! Use `dtype` instead!


✅ Model loaded fully on GPU


In [ ]:
def run_inference(
    instruction: str,
    max_new_tokens: int = 512,
    temperature: float = 0.3,
    top_p: float = 0.9,
    do_sample: bool = True,
):
    prompt = (
        "### Instruction:\n"
        f"{instruction.strip()}\n\n"
        "### Response:\n"
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        padding=False,
    ).to(DEVICE)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=do_sample,
            temperature=temperature if do_sample else 0.0,
            top_p=top_p if do_sample else 1.0,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.eos_token_id,
        )

    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # --- HARD STOP: prevent training-artifact leakage ---
    response = decoded.split("### Response:")[-1]
    response = response.split("###")[0].strip()

    return response


In [ ]:
instruction = """
A 7-year-old child presents with fever, sore throat, and a strawberry tongue.
Several days later, a sandpaper-like rash develops.
What is the diagnosis and a potential complication if untreated?
"""

print(run_inference(instruction))


The diagnosis is scarlet fever, and the potential complication is rheumatic fever.


In [ ]:
instruction = """
A 55-year-old man presents with crushing chest pain radiating to the left arm,
associated with diaphoresis and nausea. ECG shows ST elevation in leads II, III,
and aVF. What is the most likely diagnosis and underlying pathology?
"""

print(run_inference(instruction))


The most likely diagnosis is acute myocardial infarction (AMI), and the underlying pathology is atherosclerotic coronary artery disease (CAD).


In [ ]:
instruction = """
A patient with type 1 diabetes presents with abdominal pain, vomiting, and deep,
rapid breathing. Laboratory tests show a blood glucose level of 450 mg/dL.
What is the next best step in management?
"""

print(run_inference(instruction))


The next best step in management is to initiate insulin therapy and consider
possible complications such as diabetic ketoacidosis (DKA) or hyperosmolar
hyperglycemic state (HHS).


In [ ]:
instruction = """
Explain why patients with chronic kidney disease commonly develop anemia.
"""

print(run_inference(instruction))


Patients with chronic kidney disease commonly develop anemia due to a combination of factors, including reduced erythropoiesis, impaired iron absorption, and increased loss of red blood cells.


In [ ]:
instruction = """
A young woman presents with heat intolerance, weight loss, palpitations,
and tremor. Physical examination reveals exophthalmos.
What is the most likely diagnosis?
"""

print(run_inference(instruction))


In [ ]:
instruction = """
A 72-year-old man presents with exertional dyspnea and chest pain.
On physical examination, there is a crescendo–decrescendo systolic murmur
heard best at the right upper sternal border that radiates to the carotids.
What is the most likely diagnosis and underlying cause?
"""

print(run_inference(instruction))


The most likely diagnosis is aortic stenosis, and the underlying cause is aortic valve calcification.


In [ ]:
instruction = """
A patient with chronic obstructive pulmonary disease presents with worsening
shortness of breath. Arterial blood gas analysis shows hypoxemia with normal
carbon dioxide levels. What is the most likely mechanism causing hypoxemia?
"""

print(run_inference(instruction))


The most likely mechanism causing hypoxemia in this patient is hypoxemia due to
restrictive lung disease, such as interstitial lung disease or pulmonary fibrosis.


In [ ]:
instruction = """
A patient presents with weakness of the right face and arm, along with
difficulty speaking. Sensation is intact. Where is the most likely lesion?
"""

print(run_inference(instruction))


The most likely location for the lesion is the right pons.


In [ ]:
instruction = """
A patient with prolonged vomiting presents with metabolic alkalosis.
What is the primary mechanism responsible for this acid–base disturbance?
"""

print(run_inference(instruction))


The primary mechanism responsible for metabolic alkalosis in a patient with prolonged vomiting is the loss of H+ ions from the body, which is commonly seen in patients with vomiting.


In [ ]:
instruction = """
A patient presents with fatigue, weight gain, cold intolerance, and dry skin.
Laboratory tests show elevated TSH and low free T4.
What is the most likely diagnosis?
"""

print(run_inference(instruction))


Based on the patient's symptoms and laboratory findings, the most likely diagnosis is hypothyroidism.


In [ ]:
instruction = """
A patient presents with fever, headache, neck stiffness, and photophobia.
Cerebrospinal fluid analysis shows elevated protein, low glucose, and neutrophilic
pleocytosis. What is the most likely diagnosis?
"""

print(run_inference(instruction))


The most likely diagnosis is bacterial meningitis.


In [ ]:
instruction = """
A 51-year-old woman with Crohn’s disease on infliximab presented with a 2-day history of a bullous rash on her left arm, axilla, and lateral chest wall accompanied by subjective fever. Two days before presentation, she received her second dose of the recombinant adjuvant Shingrix vaccine. She denied new medications or topical products and had no prior similar rashes. Her Crohn’s disease was at baseline with intermittent loose stools. On examination, there was diffuse erythema and swelling from the midchest to the axilla and upper arm, with multiple bullae, some with central dusky areas; mucosal surfaces were spared. She was referred to dermatology and underwent punch biopsy; PCR testing of a bulla for herpes simplex virus types 1 and 2 and varicella zoster virus was negative.
Grossly, the specimen included both intraosseous and extraosseous components. Histologic examination demonstrated cords, interconnecting strands, and islands of odontogenic epithelium embedded in a cell-rich, myxoid mesenchymal stroma. The epithelial strands and cords were lined by a double layer of cuboidal cells. The islands exhibited peripheral tall columnar cells with polarized nuclei and clear, vacuolated cytoplasm surrounding central stellate reticulum-like cells. Juxtaepithelial hyalinization was noted around some islands. No hard-tissue (enamel or dentin) formation was seen. The cellularity varied, with focal hypercellular areas and other sparsely cellular, myxoid regions. A thin fibrous capsule partially surrounded the lesion. No cytologic atypia or mitotic figures were observed on multiple sections.
"""

print(run_inference(instruction))

The differential diagnosis for this case included herpes simplex virus infection, varicella zoster virus infection, and a variety of other infectious and noninfectious causes of bullous lesions. The presence of a myxoid stroma and a cell-rich,


In [ ]:
instruction = """
A 22-year-old Chinese woman was referred for evaluation of oligomenorrhea. Menarche was at age 12 with previously regular 30-day cycles. Over the past year, she noted spotting two to three times weekly. She denied weight gain, hirsutism, Cushingoid features, or thyroid symptoms. She denied medications or exogenous androgens.

On examination, blood pressure was 128/70 mm Hg, BMI 18.1. She had mild facial acne, deeper voice, and mild clitoral enlargement (10 × 5 mm); there was no hirsutism by Ferriman–Gallwey score. Abdominal, cardiac, respiratory, thyroid, and neurologic examinations were normal.

Initial laboratory evaluation showed:
– Total testosterone 5.35 nmol/L (154.3 ng/dL) [reference 0.48–1.85 nmol/L]
– 17-hydroxyprogesterone (17OHP) 18.9 nmol/L (623.7 ng/dL) [follicular phase 0.3–3.3; luteal phase 2.9–15.2 nmol/L]

A transabdominal pelvic ultrasound demonstrated a normal uterus and a left ovarian cyst measuring 18.2 × 16 mm, interpreted as a simple physiological cyst; the right ovary was normal.
"""

print(run_inference(instruction))

The recommended duration of treatment for polycystic ovary syndrome in this patient was 10 days.
